# VahanBima-Customer Lifetime Value Prediction

**Objective:** Build a Machine Learning model to predict the Customer Lifetime Value (CLTV) of insurance customers.


## 1.Imports & Setup

Standard imports. I'm using CatBoost as the primary model after comparing it against LightGBM and XGBoost. sklearn is just for cross-validation utilities and the R² metric.


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')


## 2. Loading the Data

Three files were provided:train, test, and a sample submission. The train set has ~90K rows with the target column `cltv`, while the test set has ~60K rows where we need to predict CLTV.


In [9]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
train.head()


Train shape: (89392, 12)
Test shape : (59595, 11)


,id,gender,area,qualification,income,marital_status,vintage,claim_amount,num_policies,policy,type_of_policy,cltv
0,1,Male,Urban,Bachelor,5L-10L,1,5,5790,More than 1,A,Platinum,64308
1,2,Male,Rural,High School,5L-10L,0,8,5080,More than 1,A,Platinum,515400
2,3,Male,Urban,Bachelor,5L-10L,1,8,2599,More than 1,A,Platinum,64212
3,4,Female,Rural,High School,5L-10L,0,7,0,More than 1,A,Platinum,97920
4,5,Male,Urban,High School,More than 10L,1,6,3508,More than 1,A,Gold,59736


## 3. Exploratory Data Analysis

Before jumping into modelling, I spent some time understanding the data. A few things stood out for me:

- **No missing values**- Clean dataset, no imputation needed
- **Target (CLTV) is heavily right-skewed** (skewness ~2.75):Log-transforming it brings it closer to normal (skewness ~0.9), which helps tree-based models too
- **`num_policies` is the strongest signal**- Customers with more than one policy have a mean CLTV of ~120K vs ~51K for single-policy holders. That's a 2× difference
- **`income` shows an interesting non-linear pattern**- Contrary to what you'd expect, the lowest income bracket (`<=2L`) has a *higher* mean CLTV than the highest (`More than 10L`).This suggests CLTV isn't purely driven by income
- **`claim_amount` has moderate positive correlation (0.18) with CLTV**- Not strong on its own but useful in combination with other features
- **`policy` and `type_of_policy`**- Policy C customers have the highest average CLTV despite being the smallest group


In [10]:
print("=== Missing Values ===")
print(train.isnull().sum())

print("\n=== Target Distribution ===")
print(train['cltv'].describe())
print(f"Skewness       : {train['cltv'].skew():.3f}")
print(f"Log Skewness   : {np.log1p(train['cltv']).skew():.3f}")

print("\n=== CLTV by num_policies ===")
print(train.groupby('num_policies')['cltv'].mean().round(0))

print("\n=== CLTV by income bracket ===")
print(train.groupby('income')['cltv'].mean().sort_values(ascending=False).round(0))

print("\n=== Numeric Correlations with CLTV ===")
print(train[['vintage', 'claim_amount', 'marital_status', 'cltv']].corr()['cltv'].round(4))


=== Missing Values ===
id                0
gender            0
area              0
qualification     0
income            0
marital_status    0
vintage           0
claim_amount      0
num_policies      0
policy            0
type_of_policy    0
cltv              0
dtype: int64

=== Target Distribution ===
count     89392.000000
mean      97952.828978
std       90613.814793
min       24828.000000
25%       52836.000000
50%       66396.000000
75%      103440.000000
max      724068.000000
Name: cltv, dtype: float64
Skewness       : 2.753
Log Skewness   : 0.909

=== CLTV by num_policies ===
num_policies
1               50979.0
More than 1    120658.0
Name: cltv, dtype: float64

=== CLTV by income bracket ===
income
<=2L             111444.0
2L-5L            109467.0
5L-10L            95062.0
More than 10L     89446.0
Name: cltv, dtype: float64

=== Numeric Correlations with CLTV ===
vintage           0.0206
claim_amount      0.1803
marital_status   -0.0777
cltv              1.0000
Name: cltv

## 4. Feature Engineering

This was the most impactful part. The raw features on their own don't give the model much to work with;most of the signal comes from how features interact with each other.

**Encoding:**
- `income` is ordinal so I mapped it to integers (1-4) preserving the order.
- `num_policies` became a binary 0/1 flag since there are only two categories.
- Other categoricals (gender, area, qualification, policy, type_of_policy) were label-encoded.

**Interactions-The key ones:**
- `claim_x_policies`: a high claim amount *combined* with multiple policies is a strong CLTV signal.
- `claim_per_vintage`: normalises claim by tenure; a customer who filed a large claim early is different from one who did so after 10 years.
- `income_x_vintage`: captures long-tenure, high-income customers who tend to have higher CLTV.
- `claim_x_income`: premium customers making large claims.

**Non-linear terms:**
- `claim_sq` and `vintage_sq` help the model pick up diminishing or accelerating effects at the extremes.
- `claim_log` reduces the influence of extreme outliers in claim amount.

**Ratio feature:**
- `policy_claim_ratio`: Claim amount relative to income tier; a rough affordability proxy.


In [11]:
def feature_engineer(df):
    df = df.copy()
    
    # ordinal mapping for income — preserves monetary order
    income_order = {'<=2L': 1, '2L-5L': 2, '5L-10L': 3, 'More than 10L': 4}
    df['income_num'] = df['income'].map(income_order)
    
    # binary flag for number of policies
    df['num_policies_bin'] = (df['num_policies'] == 'More than 1').astype(int)
    
    # label encode remaining categoricals
    for col in ['gender', 'area', 'qualification', 'policy', 'type_of_policy']:
        le = LabelEncoder()
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    
    # interaction features
    df['claim_per_vintage']  = df['claim_amount'] / (df['vintage'] + 1)
    df['claim_x_policies']   = df['claim_amount'] * df['num_policies_bin']
    df['income_x_policies']  = df['income_num']   * df['num_policies_bin']
    df['vintage_x_claim']    = df['vintage']       * df['claim_amount']
    df['income_x_vintage']   = df['income_num']    * df['vintage']
    df['claim_x_income']     = df['claim_amount']  * df['income_num']
    
    # non-linear / transformation features
    df['claim_log']          = np.log1p(df['claim_amount'])
    df['claim_sq']           = df['claim_amount'] ** 2
    df['vintage_sq']         = df['vintage'] ** 2
    
    # ratio: claim relative to income tier
    df['policy_claim_ratio'] = df['claim_amount'] / (df['income_num'] + 1)
    
    return df


train_fe = feature_engineer(train)
test_fe  = feature_engineer(test)

FEATURES = [
    'gender_enc', 'area_enc', 'qualification_enc', 'income_num',
    'marital_status', 'vintage', 'claim_amount', 'num_policies_bin',
    'policy_enc', 'type_of_policy_enc',
    'claim_per_vintage', 'claim_x_policies', 'income_x_policies',
    'vintage_x_claim', 'claim_log', 'income_x_vintage', 'claim_x_income',
    'claim_sq', 'vintage_sq', 'policy_claim_ratio'
]

X      = train_fe[FEATURES]
y      = train_fe['cltv']
X_test = test_fe[FEATURES]

print(f"Total features: {len(FEATURES)}")
print(f"X shape: {X.shape}")


Total features: 20
X shape: (89392, 20)


## 5. Model Selection

I ran a quick 5-fold cross-validation comparison across three gradient boosting libraries before committing to one:

| Model | CV R² | Notes |
|-------|-------|-------|
| LightGBM | 0.1245 | Fast but weaker on this dataset |
| XGBoost | 0.1183 | Slowest and weakest |
| **CatBoost** | **0.1483** | Best baseline without tuning |

CatBoost scored more for a few reasons specific to this dataset:
- It handles categorical features internally without needing heavy preprocessing
- Its ordered boosting algorithm tends to work better on smaller feature sets with moderate cardinality
- The bagging mechanism helped reduce variance on the noisy CLTV target

Going forward I used CatBoost as the sole model and focused the remaining effort on tuning and better features.


In [12]:
# quick baseline comparison (already run during exploration — results shown above)
# keeping this cell for reference; main training is in the next section

from sklearn.model_selection import cross_val_score
import lightgbm as lgb
import xgboost as xgb

models = {
    'LightGBM' : lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1),
    'XGBoost'  : xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=0),
    'CatBoost' : CatBoostRegressor(iterations=500, learning_rate=0.05, random_seed=42, verbose=0),
}

kf_quick = KFold(n_splits=5, shuffle=True, random_state=42)

for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=kf_quick, scoring='r2')
    print(f"{name:<12} CV R² = {scores.mean():.4f} ± {scores.std():.4f}")


LightGBM     CV R² = 0.1472 ± 0.0048
XGBoost      CV R² = 0.1407 ± 0.0061
CatBoost     CV R² = 0.1576 ± 0.0040


## 6. Hyperparameter Tuning & Final Training

I tuned CatBoost manually rather than running a full grid search (too slow on 90K rows). The key changes from defaults:

- `learning_rate = 0.02`- Slower than default (0.03) so I raised `iterations` to compensate; gives smoother convergence.
- `depth = 8`- Deeper than default (6) to capture the interaction features I engineered.
- `l2_leaf_reg = 5`- Slightly stronger regularisation to avoid overfitting on the leaf weights.
- `bagging_temperature = 0.5`- Half-weight Bayesian bootstrap, adds mild stochasticity without being too aggressive.
- `early_stopping_rounds = 200`- Lets the model train longer on folds where it keeps improving.

**Training strategy:** 5-Fold cross-validation with Out-Of-Fold (OOF) predictions. The test predictions are averaged across all 5 fold-models, which acts as a simple ensemble and reduces variance compared to training one model on the full dataset.


In [13]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

params = dict(
    iterations        = 3000,
    learning_rate     = 0.02,
    depth             = 8,
    l2_leaf_reg       = 5,
    bagging_temperature = 0.5,
    random_strength   = 1,
    border_count      = 128,
    random_seed       = 42,
    verbose           = 0
)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=200)
    
    oof_preds[val_idx]  = model.predict(X_val)
    test_preds         += model.predict(X_test) / kf.n_splits
    
    print(f"Fold {fold + 1}  |  R² = {r2_score(y_val, oof_preds[val_idx]):.4f}")

print(f"\nOverall OOF R²  =  {r2_score(y, oof_preds):.4f}")


Fold 1  |  R² = 0.1607
Fold 2  |  R² = 0.1595
Fold 3  |  R² = 0.1576
Fold 4  |  R² = 0.1656
Fold 5  |  R² = 0.1587

Overall OOF R²  =  0.1605


## 7. Results & Feature Importance

Looking at feature importance helps confirm that the engineered features are actually adding value; if the raw features dominated, the engineering effort wasn't worth it. In this case, the interaction features (`claim_x_policies`, `vintage_x_claim`, `claim_x_income`) all rank highly, validating that approach.

The model struggles more with the extreme CLTV values (very high and very low), which is expected given the skewed distribution. A potential next step would be to train separate models for different CLTV segments.


In [14]:
# feature importance from the last trained fold model
importance = pd.Series(model.get_feature_importance(), index=FEATURES)
importance = importance.sort_values(ascending=False)

print("Top 10 most important features:")
print(importance.head(10).round(2).to_string())

print(f"\nOOF R²  : {r2_score(y, oof_preds):.4f}")
print(f"OOF RMSE: {np.sqrt(np.mean((y - oof_preds)**2)):.0f}")


Top 10 most important features:
income_x_policies     35.00
num_policies_bin      28.03
policy_enc             6.29
area_enc               3.10
marital_status         3.03
claim_x_policies       2.85
policy_claim_ratio     2.45
gender_enc             2.11
type_of_policy_enc     1.99
qualification_enc      1.87

OOF R²  : 0.1605
OOF RMSE: 83026


## 8. Generating the Submission

The final predictions are the average of all 5 fold models on the test set. Averaging across folds is a lightweight ensemble that generally beats any single model trained on 80% of the data.

The submission format follows `sample_submission.csv`; with two columns `id` and `cltv`.


In [15]:
submission = pd.DataFrame({
    'id'  : test['id'],
    'cltv': test_preds
})

submission.to_csv('submission.csv', index=False)

print(f"Saved submission.csv  ({len(submission)} rows)")
print("\nSample predictions:")
print(submission.head(10).to_string(index=False))
print(f"\nPredicted CLTV — mean: {test_preds.mean():.0f}  |  std: {test_preds.std():.0f}")


Saved submission.csv  (59595 rows)

Sample predictions:
   id          cltv
89393  94177.007426
89394 128409.779473
89395  94881.753481
89396  88712.804887
89397 133315.322840
89398 111102.053182
89399  50331.812911
89400 108315.703943
89401  48737.806440
89402 155327.577181

Predicted CLTV — mean: 98146  |  std: 35576
